In [1]:
import json
import pandas as pd
import ast

In [2]:
df = pd.read_csv('bitcoin_tweet_2022.csv')
df

,date,text2,hashtags
0,2022-01-14,death cross bitcoin dump,['bitcoin']
1,2022-01-14,teaser bitcoin cryptocurrency furniture,"['bitcoin', 'cryptocurrency', 'furniture']"
2,2022-01-14,well time bitcoin mocked made fun upon careful,['bitcoin']
3,2022-01-14,bitcoin christmas ethereum shibthis opportunit...,"['bitcoin', 'christmas', 'ethereum', 'shib', '..."
4,2022-01-14,ainu token coming soon play earntelegram ainu ...,"['ainutoken', 'playtoearn', 'nfts', 'metaverse..."
...,...,...,...
839594,2022-12-24,making noise crypto bsc ethereum bitcoin alt c...,"['crypto', 'bsc', 'ethereum', 'bitcoin', 'alt']"
839595,2022-12-24,russiaukraine war reason biggest bitcoin sello...,['bitcoin']
839596,2022-12-24,glassnodealerts bitcoin btc percent supply las...,['bitcoin']
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,"['bitcoin', 'decred', 'btc', 'dcr']"


In [3]:
df['hashtags'] = df['hashtags'].str.lower()
df['hashtags'] = df['hashtags'].apply(ast.literal_eval)

In [4]:
df_expanded = df.explode('hashtags')
df_expanded

,date,text2,hashtags
0,2022-01-14,death cross bitcoin dump,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,cryptocurrency
1,2022-01-14,teaser bitcoin cryptocurrency furniture,furniture
2,2022-01-14,well time bitcoin mocked made fun upon careful,bitcoin
...,...,...,...
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,decred
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,btc
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,dcr


In [24]:
threshold = 280
hashtag_counts = df_expanded['hashtags'].value_counts()
valid_hashtags = hashtag_counts[hashtag_counts >= threshold].index

df_filter = df_expanded[df_expanded['hashtags'].isin(valid_hashtags)]
df_filter = df_filter[['date','hashtags']]
with open("hashtag_counts.txt", "w") as f:
    f.write(df_filter['hashtags'].value_counts().to_string())
len(df_filter['hashtags'].unique())

1020

In [26]:
df_filter['date'] = pd.to_datetime(df_filter['date'])
df_filter

,date,hashtags
0,2022-01-14,bitcoin
1,2022-01-14,bitcoin
1,2022-01-14,cryptocurrency
2,2022-01-14,bitcoin
3,2022-01-14,bitcoin
...,...,...
839595,2022-12-24,bitcoin
839596,2022-12-24,bitcoin
839597,2022-12-24,bitcoin
839597,2022-12-24,btc


In [27]:
from tqdm import tqdm

full_dates = pd.date_range(start=df_filter['date'].min(), end=df_filter['date'].max())
hashtags = df_filter['hashtags'].unique()
result = []

for tag in tqdm(hashtags, desc="Processing hashtags"):
    df_tag = df_filter[df_filter['hashtags'] == tag]
    daily_counts = df_tag.groupby('date').size()
    daily_counts = daily_counts.reindex(full_dates, fill_value=0)

    timeline = [
        {"date": date.strftime('%Y-%m-%d'), "count": int(count)}
        for date, count in daily_counts.items()
    ]

    result.append({
        "hashtag": tag,
        "timeline": timeline
    })


Processing hashtags: 100%|██████████| 1020/1020 [03:12<00:00,  5.30it/s]


In [28]:

with open("hashtag_timeline_1000.json", "w") as f:
    json.dump(result, f, indent=4)